# Notebook 05 — Assembly & Validation
Merge real + TabSyn + GReaT into the final **1.2M session HoneySynth-1M** dataset.
Run full quality validation (Wasserstein, adversarial AUC, TSTR) before training.

| Source | Target | Actual |
|--------|--------|--------|
| Real Cowrie | 180,000 (15%) | from Notebook 01/02 |
| Real Transfer (CIC/UNSW) | 60,000 (5%) | from Notebook 01/02 |
| TabSyn synthetic | 720,000 (60%) | from Notebook 03 |
| GReaT synthetic | 240,000 (20%) | from Notebook 04 |
| **Total** | **1,200,000** | |

In [ ]:
import sys, logging, warnings, json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger()

ROOT       = Path("..").resolve()
DATA_PROC  = ROOT / "data" / "processed"
DATA_SYNTH = ROOT / "data" / "synthetic"
DATA_FINAL = ROOT / "data" / "final"
DATA_FINAL.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT))

from configs.schema import N_FEATURES, N_CLASSES, IDX_TO_LABEL, SPLIT_TARGETS

## 5.1 Load All Sources

In [ ]:
X_real   = np.load(DATA_PROC  / "X_real.npy")
y_real   = np.load(DATA_PROC  / "y_real.npy")
X_tabsyn = np.load(DATA_SYNTH / "X_tabsyn.npy")
y_tabsyn = np.load(DATA_SYNTH / "y_tabsyn.npy")
X_great  = np.load(DATA_SYNTH / "X_great.npy")
y_great  = np.load(DATA_SYNTH / "y_great.npy")

print("Loaded sources:")
print(f"  Real:    X={X_real.shape}    y={y_real.shape}")
print(f"  TabSyn:  X={X_tabsyn.shape}  y={y_tabsyn.shape}")
print(f"  GReaT:   X={X_great.shape}   y={y_great.shape}")
print(f"\n  TOTAL: {len(X_real)+len(X_tabsyn)+len(X_great):,} sessions")
print(f"  TARGET: {sum(SPLIT_TARGETS.values()):,} sessions")

## 5.2 Resample to Exact Split Targets

In [ ]:
from sklearn.utils import resample

def resample_to_target(X, y, target_n, seed=42):
    n = len(X)
    if n == target_n:
        return X, y
    elif n > target_n:
        idx = np.random.RandomState(seed).choice(n, target_n, replace=False)
        return X[idx], y[idx]
    else:
        idx = np.random.RandomState(seed).choice(n, target_n, replace=True)
        return X[idx], y[idx]

# Real data: keep as-is up to target (real + transfer combined = 240k target)
real_target = SPLIT_TARGETS["real_cowrie"] + SPLIT_TARGETS["real_transfer"]
X_real_f, y_real_f = resample_to_target(X_real, y_real, real_target, seed=1)

X_tabsyn_f, y_tabsyn_f = resample_to_target(
    X_tabsyn, y_tabsyn, SPLIT_TARGETS["tabsyn_synth"], seed=2)

X_great_f, y_great_f = resample_to_target(
    X_great, y_great, SPLIT_TARGETS["great_synth"], seed=3)

print(f"Resampled to targets:")
print(f"  Real:    {len(X_real_f):,} (target {real_target:,})")
print(f"  TabSyn:  {len(X_tabsyn_f):,} (target {SPLIT_TARGETS['tabsyn_synth']:,})")
print(f"  GReaT:   {len(X_great_f):,} (target {SPLIT_TARGETS['great_synth']:,})")

## 5.3 Merge + Source Tags

In [ ]:
X_full = np.vstack([X_real_f, X_tabsyn_f, X_great_f]).astype(np.float32)
y_full = np.concatenate([y_real_f, y_tabsyn_f, y_great_f]).astype(np.int64)
source_full = np.array(
    ["real"] * len(X_real_f) +
    ["synthetic"] * (len(X_tabsyn_f) + len(X_great_f))
)
generator_full = np.array(
    ["cowrie"] * len(X_real_f) +
    ["tabsyn"] * len(X_tabsyn_f) +
    ["great"]  * len(X_great_f)
)

print(f"Merged dataset: X={X_full.shape}  y={y_full.shape}")
assert len(X_full) == sum(SPLIT_TARGETS.values()), "Total doesn't match target!"
print(f"✓ Total matches target: {len(X_full):,}")

# Shuffle once (keep alignment across all arrays)
rng = np.random.RandomState(42)
perm = rng.permutation(len(X_full))
X_full, y_full = X_full[perm], y_full[perm]
source_full, generator_full = source_full[perm], generator_full[perm]

## 5.4 Stratified Split — Train / Val / Test(synth) / Test(real, TSTR)

In [ ]:
def safe_stratified_split(X, y, test_size, seed=42):
    counts = np.bincount(y, minlength=N_CLASSES)
    can_stratify = bool((counts[counts > 0] >= 2).all())
    return train_test_split(
        X, y, test_size=test_size,
        stratify=y if can_stratify else None,
        random_state=seed,
    )

real_mask  = source_full == "real"
synth_mask = source_full == "synthetic"

X_r, y_r = X_full[real_mask],  y_full[real_mask]
X_s, y_s = X_full[synth_mask], y_full[synth_mask]

# Held-out REAL test set → TSTR evaluation (the paper's key metric)
n_test_real = min(60_000, int(len(X_r) * 0.30))
X_r_train, X_test_real, y_r_train, y_test_real = safe_stratified_split(
    X_r, y_r, test_size=n_test_real, seed=11)

# Held-out SYNTHETIC test set → standard evaluation
n_test_synth = 60_000
X_s_train, X_test_synth, y_s_train, y_test_synth = safe_stratified_split(
    X_s, y_s, test_size=n_test_synth, seed=12)

# Combine remaining for train/val pool
X_pool = np.vstack([X_r_train, X_s_train])
y_pool = np.concatenate([y_r_train, y_s_train])

X_train, X_val, y_train, y_val = safe_stratified_split(
    X_pool, y_pool, test_size=0.10, seed=13)

print("Final splits:")
print(f"  Train:        {len(X_train):>9,}")
print(f"  Val:          {len(X_val):>9,}")
print(f"  Test (synth): {len(X_test_synth):>9,}")
print(f"  Test (real):  {len(X_test_real):>9,}  ← TSTR evaluation")
print(f"  {'─'*30}")
print(f"  TOTAL:        {len(X_train)+len(X_val)+len(X_test_synth)+len(X_test_real):>9,}")

## 5.5 Normalise — Fit on Train Only

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train)

X_train_s      = scaler.transform(X_train).astype(np.float32)
X_val_s        = scaler.transform(X_val).astype(np.float32)
X_test_synth_s = scaler.transform(X_test_synth).astype(np.float32)
X_test_real_s  = scaler.transform(X_test_real).astype(np.float32)

joblib.dump(scaler, DATA_FINAL / "feature_scaler.pkl")
print(f"Scaler fitted on {len(X_train):,} training rows.")
print(f"Mean range: [{scaler.mean_.min():.3f}, {scaler.mean_.max():.3f}]")
print(f"Scale range: [{scaler.scale_.min():.3f}, {scaler.scale_.max():.3f}]")

## 5.6 Save Final Splits

In [ ]:
splits = {
    "X_train": X_train_s,      "y_train": y_train,
    "X_val":   X_val_s,        "y_val":   y_val,
    "X_test_synth": X_test_synth_s, "y_test_synth": y_test_synth,
    "X_test_real":  X_test_real_s,  "y_test_real":  y_test_real,
}
for name, arr in splits.items():
    np.save(DATA_FINAL / f"{name}.npy", arr)
    print(f"  Saved {name}.npy  {arr.shape}")

print(f"\n✓ All splits saved to {DATA_FINAL}")

## 5.7 Full Quality Validation
This produces every number for **Table 1** of the paper.

In [ ]:
from src.validators.quality_checks import run_full_validation

# Use UNSCALED data for Wasserstein/adversarial checks (interpretable units)
report = run_full_validation(
    X_real=X_r_train[:30_000],  y_real=y_r_train[:30_000],
    X_synth=X_s_train[:30_000], y_synth=y_s_train[:30_000],
    output_path=DATA_FINAL / "quality_report.json",
)

print(f"\n{'='*50}")
print(f"VERDICT: {report['verdict']}")
print(f"{'='*50}")

## 5.8 Kill-Chain Violation Rate (KCVR) Baseline

In [ ]:
from src.generators.kill_chain_simulator import compute_kcvr

# Load metadata with sequences for KCVR computation
meta_frames = []
real_meta = DATA_PROC / "real_meta.parquet"
tabsyn_meta = DATA_SYNTH / "tabsyn_meta.parquet"
great_meta = DATA_SYNTH / "great_meta.parquet"

for p, seq_col in [(tabsyn_meta, "micro_state_sequence" if False else None),
                    (great_meta, "micro_state_sequence")]:
    if p.exists():
        df_m = pd.read_parquet(p)
        if "micro_state_sequence" in df_m.columns:
            meta_frames.append(df_m)

if meta_frames:
    df_seq = pd.concat(meta_frames, ignore_index=True)
    kcvr = compute_kcvr(df_seq, seq_col="micro_state_sequence")
    print(f"Dataset Kill-Chain Violation Rate (post-filter): {kcvr:.4f}")
    print(f"  (Target: 0.0000 — all generated sequences are DAG-valid)")
else:
    print("No sequence metadata found for KCVR computation")

## 5.9 Dataset Card Summary — Table 1 for Paper

In [ ]:
from configs.schema import N_FEATURES, N_CLASSES, FEATURE_GROUPS

summary = {
    "dataset_name":          "HoneySynth-1M",
    "total_sessions":        len(X_train)+len(X_val)+len(X_test_synth)+len(X_test_real),
    "real_sessions":         int(real_mask.sum()),
    "real_pct":              round(real_mask.sum()/len(X_full)*100, 1),
    "tabsyn_sessions":       len(X_tabsyn_f),
    "tabsyn_pct":            round(len(X_tabsyn_f)/len(X_full)*100, 1),
    "great_sessions":        len(X_great_f),
    "great_pct":             round(len(X_great_f)/len(X_full)*100, 1),
    "n_features":            N_FEATURES,
    "n_classes":             N_CLASSES,
    "n_feature_groups":      len(FEATURE_GROUPS),
    "mitre_techniques":      45,
    "mitre_tactics":         9,
    "min_samples_per_class": int(np.bincount(y_full, minlength=N_CLASSES).min()),
    "max_samples_per_class": int(np.bincount(y_full, minlength=N_CLASSES).max()),
    "adversarial_auc":       report["adversarial_auc"]["adversarial_auc"],
    "tstr_macro_f1":         report["tstr"]["tstr_macro_f1"],
    "privacy_note":          "DP-SGD recommended for public release (epsilon=10, delta=1e-6)",
}

print("="*60)
print("  TABLE 1 — Dataset Statistics (HoneySynth-1M)")
print("="*60)
for k, v in summary.items():
    print(f"  {k:<25} {v}")
print("="*60)

with open(DATA_FINAL / "dataset_card.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"\nDataset card saved → {DATA_FINAL}/dataset_card.json")
print("\n✓ DATASET ASSEMBLY COMPLETE — ready for Phase 5 (model training)")